# Chatbot del peaje en RunPod — Qwen 2.5 (Ollama)

> **© 2026 Matías Darío Mercier Rengel** — Sistema Inteligente de Control de Peaje (IAM-811, UNIFRANZ). Todos los derechos reservados. Repositorio original: https://github.com/MATT-UwU/Laboratorio-final-Hito-4

Este Pod corre **solo el LLM del agente conversacional**. La inferencia de placas/vehículos sigue en Colab.
No hace falta subir otros archivos: este notebook crea todo. Ejecuta las celdas en orden (Jupyter Lab del Pod).

Requisito del Pod: puerto **HTTP 8000** expuesto (Edit Pod → Expose HTTP Ports → 8000).


In [ ]:
# 1. GPU
!nvidia-smi


In [ ]:
# 2. Instalar Ollama y dependencias (los modelos quedan en /workspace/ollama)
import os, subprocess
os.environ['OLLAMA_MODELS'] = '/workspace/ollama'
os.makedirs('/workspace/ollama', exist_ok=True)
subprocess.run('apt-get update -qq && apt-get install -y -qq zstd curl > /dev/null', shell=True)
if subprocess.run('command -v ollama', shell=True).returncode != 0:
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh', shell=True, check=True)
subprocess.run('pip install -q fastapi==0.115.6 "uvicorn[standard]==0.34.0"', shell=True, check=True)
print('✓ Ollama y FastAPI instalados')


In [ ]:
# 3. Arrancar Ollama y descargar Qwen 2.5 7B (~4,7 GB la primera vez)
import time, urllib.request
ollama = subprocess.Popen(['ollama', 'serve'], env=dict(os.environ), stdout=open('/workspace/ollama.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen('http://127.0.0.1:11434/api/tags', timeout=2); break
    except Exception:
        time.sleep(1)
subprocess.run(['ollama', 'pull', 'qwen2.5:7b'], check=True)
print('✓ Qwen listo')


In [ ]:
# 4. Servidor del chatbot (FastAPI) protegido con token, en el puerto 8000
import getpass
from pathlib import Path
Path('/workspace/llm_servidor.py').write_text(r'''# © 2026 Matías Darío Mercier Rengel — Chatbot (Qwen) del Sistema Inteligente de Control de Peaje. Todos los derechos reservados.
"""
Servidor del LLM del agente conversacional (Qwen vía Ollama) para RunPod.

Solo expone el chatbot: la inferencia de placas/vehículos sigue en Colab (INFERENCE_URL).
Ollama escucha en 127.0.0.1:11434; desde Internet se accede por este servidor, protegido con X-API-Token.

    LLM_API_TOKEN=... uvicorn llm_servidor:app --host 0.0.0.0 --port 8000
"""
import hmac
import json
import os
import time
import urllib.request
from typing import Any, Dict, Optional

from fastapi import FastAPI, Header, HTTPException

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
LLM_MODELO = os.environ.get("LLM_MODELO", "qwen2.5:7b")
LLM_API_TOKEN = os.environ.get("LLM_API_TOKEN", "")

app = FastAPI(title="LLM del agente del peaje (Qwen)", version="1.0.0")


def _ollama(ruta: str, cuerpo: Optional[dict] = None, timeout: int = 120) -> dict:
    datos = None if cuerpo is None else json.dumps(cuerpo).encode("utf-8")
    req = urllib.request.Request(
        OLLAMA_URL.rstrip("/") + ruta, data=datos,
        headers={"Content-Type": "application/json"}, method="POST" if datos else "GET",
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))


def _verificar(token: Optional[str]) -> None:
    if not LLM_API_TOKEN or not token or not hmac.compare_digest(token, LLM_API_TOKEN):
        raise HTTPException(status_code=401, detail="Token inválido")


@app.get("/")
def raiz():
    try:
        modelos = [m.get("name") for m in _ollama("/api/tags", timeout=5).get("models", [])]
        listo = any(m.startswith(LLM_MODELO) for m in modelos)
    except Exception:
        modelos, listo = [], False
    return {"estado": "activo", "servicio": "llm", "autor": "Matías Darío Mercier Rengel", "modelo": LLM_MODELO, "modelo_disponible": listo}


@app.get("/llm/estado")
def llm_estado(x_api_token: Optional[str] = Header(default=None)):
    _verificar(x_api_token)
    return raiz()


@app.post("/llm/chat")
def llm_chat(cuerpo: Dict[str, Any], x_api_token: Optional[str] = Header(default=None)):
    """Conversación con herramientas (function calling) en el formato /api/chat de Ollama."""
    _verificar(x_api_token)
    t0 = time.perf_counter()
    try:
        resp = _ollama("/api/chat", {
            "model": cuerpo.get("model") or LLM_MODELO,
            "messages": cuerpo.get("messages", []),
            "tools": cuerpo.get("tools", []),
            "stream": False,
            "options": {"temperature": 0.1},
        })
    except Exception as e:
        raise HTTPException(status_code=503, detail=f"LLM no disponible: {e}")
    resp["latencia_ms"] = round((time.perf_counter() - t0) * 1000, 2)
    return resp
''')
token = getpass.getpass('LLM_API_TOKEN (no se mostrará): ').strip()
env = dict(os.environ, LLM_API_TOKEN=token, LLM_MODELO='qwen2.5:7b')
servidor = subprocess.Popen(['uvicorn', 'llm_servidor:app', '--host', '0.0.0.0', '--port', '8000'],
                            cwd='/workspace', env=env, stdout=open('/workspace/llm_servidor.log', 'w'), stderr=subprocess.STDOUT)
time.sleep(5)
print(open('/workspace/llm_servidor.log').read()[-1500:])


In [ ]:
# 5. Prueba local y URL pública para el VPS
import json
print(json.loads(urllib.request.urlopen('http://127.0.0.1:8000/', timeout=10).read()))
req = urllib.request.Request('http://127.0.0.1:8000/llm/chat', method='POST',
      headers={'X-API-Token': token, 'Content-Type': 'application/json'},
      data=json.dumps({'messages': [{'role': 'user', 'content': 'Responde solo: listo'}]}).encode())
r = json.loads(urllib.request.urlopen(req, timeout=120).read())
print('Qwen dice:', r['message']['content'], '| latencia', r['latencia_ms'], 'ms')
print('LLM_URL=https://' + os.environ.get('RUNPOD_POD_ID', '<POD_ID>') + '-8000.proxy.runpod.net')


Copia la línea **LLM_URL** y pásala para configurarla en el VPS. Deja el Pod encendido durante la demo y la revisión. Logs: `/workspace/llm_servidor.log`, `/workspace/ollama.log`.